In [1]:
! pip install psycopg2 python-dotenv

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   --------------- ------------------------ 1.0/2.8 MB 5.6 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.8 MB 5.6 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.8 MB 5.6 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.8 MB 5.6 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 1.8 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 1.8 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 1.8 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 1.8 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 1.8 MB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 1.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import csv
import psycopg2
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format= "%(asctime)s %(levelname)s %(message)s"
)

In [54]:
with psycopg2.connect(
    host = os.getenv("DB_HOST"),
    database = os.getenv("DB_NAME"),
    user = os.getenv("DB_USER"),
    password = os.getenv("DB_PASS"),
    port = 5432
) as conn:
    with conn.cursor() as cursor:
        cursor.execute('''drop table clean_dealer''')
        # result = cursor.fetchall()
        # print(result)

In [37]:
def read_csv(file_path):
    '''Reads a csv file and returns a list of rows in dictionary format'''
    rows_list = []
    try:
        logging.info(f"Reading data from {file_path}")
        with open(file_path,newline="") as f: 
            reader = csv.DictReader(f)
            for row in reader:
                rows_list.append(row)
        logging.info(f"Successfully read {len(rows_list)} rows")
        return rows_list
    except Exception as e: 
        logging.error(f"Reading failed: {e}")
        raise

In [49]:
def create_clean_dealer_table():
    '''Creates clean dealer table in Database'''
    logging.info("Connecting to Database")
    try:
        with psycopg2.connect(
            host = os.getenv("DB_HOST"),
            database = os.getenv("DB_NAME"),
            user = os.getenv("DB_USER"),
            password = os.getenv("DB_PASS"),
            port = 5432
        ) as conn:
            logging.info("Connection Successful")
            with conn.cursor() as cursor:
                query = '''
                        CREATE TABLE clean_dealer(
                            dealer_id INT PRIMARY KEY,
                            dealer_code CHAR(10),
                            dealer_name VARCHAR(50),
                            city VARCHAR(30),
                            state CHAR(2),
                            region VARCHAR(5),
                            dealer_type VARCHAR(9),
                            created_date DATE,
                            is_active VARCHAR(5),
                            email TEXT,
                            phone TEXT,
                            credit_terms_days INT
                        )
                    '''
                logging.info(f"Executing Query | {query}")
                try:
                    cursor.execute(query)
                    logging.info(f"Table created successfully")
                except Exception as e: 
                    logging.error(f"Table creation failed: {e}")
                    raise
    except Exception as e: 
        logging.error(f"Connection failed: {e}")
        raise 

In [53]:
def insert_rows_into_clean_dealer_table(rows_list):
    '''Inserts rows into clean dealer table'''
    logging.info("Connecting to Database")
    try:
        with psycopg2.connect(
            host = os.getenv("DB_HOST"),
            database = os.getenv("DB_NAME"),
            user = os.getenv("DB_USER"),
            password = os.getenv("DB_PASS"),
            port = 5432
        ) as conn:
            logging.info("Connection Successful")
            with conn.cursor() as cursor:
                logging.info(f"Executing Table Population Query")
                insert_count = 0
                for row in rows_list:
                    try:
                        cursor.execute('''
                            INSERT INTO clean_dealer VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
                        ''',tuple(row.values()))
                        insert_count += 1
                    except Exception as e:
                        logging.warning(f"Row insertion failed: {e} | {row}")
                insert_pct = round(100.0*insert_count/len(rows_list),2)
                logging.info(f"{insert_count} rows inserted successfully ({insert_pct} % of total)")
                failed_count = len(rows_list)-insert_count
                if failed_count > 0:
                    failed_pct = round(100.0*failed_count/len(rows_list),2)
                    logging.info(f"Failed to insert {len(rows_list)-insert_count} rows ({failed_pct} % of total)")
    except Exception as e: 
        logging.error(f"Connection failed: {e}") 
        raise
        

In [35]:
def main():
    # File Paths
    root_path = "C:/Users/KIIT/Desktop/Stratlytics/02_Bootcamp/04_Python/"
    file_path = root_path + "01_Data/clean/dealer.csv"
    # Read file
    rows_list = read_csv(file_path)
    # Create table
    create_clean_dealer_table()
    # Insert rows
    insert_rows_into_clean_dealer_table(rows_list)
    

In [55]:
main()

2026-07-25 21:57:29,143 INFO Reading data from C:/Users/KIIT/Desktop/Stratlytics/02_Bootcamp/04_Python/01_Data/clean/dealer.csv
2026-07-25 21:57:29,144 INFO Successfully read 80 rows
2026-07-25 21:57:29,145 INFO Connecting to Database
2026-07-25 21:57:29,191 INFO Connection Successful
2026-07-25 21:57:29,192 INFO Executing Query | 
                        CREATE TABLE clean_dealer(
                            dealer_id INT PRIMARY KEY,
                            dealer_code CHAR(10),
                            dealer_name VARCHAR(50),
                            city VARCHAR(30),
                            state CHAR(2),
                            region VARCHAR(5),
                            dealer_type VARCHAR(9),
                            created_date DATE,
                            is_active VARCHAR(5),
                            email TEXT,
                            phone TEXT,
                            credit_terms_days INT
                        )
                

In [20]:
create_clean_dealer_table()

2026-07-25 21:16:41,665 INFO Connecting to Database
2026-07-25 21:16:41,733 INFO Connection Successful
2026-07-25 21:16:41,734 INFO Executing Table Creation Query
2026-07-25 21:16:41,789 INFO Table Created


In [24]:
query = "SELECT * FROM dealer;"
execute_query(query,"one")

2026-07-24 15:24:06,129 INFO Connecting to Database
2026-07-24 15:24:06,158 INFO Connection Successful
2026-07-24 15:24:06,160 INFO Executing Query | SELECT * FROM dealer;
2026-07-24 15:24:06,163 INFO Query Execution Successful
2026-07-24 15:24:06,163 INFO Result:
(1001, 'Prime Motors', 'West')


(1001, 'Prime Motors', 'West')

In [27]:
query = '''CREATE TABLE temp(
temp_id INT,
temp_name VARCHAR(30))'''
execute_query(query)

2026-07-24 15:26:43,744 INFO Connecting to Database
2026-07-24 15:26:43,771 INFO Connection Successful
2026-07-24 15:26:43,773 INFO Executing Query | CREATE TABLE temp(
temp_id INT,
temp_name VARCHAR(30))
2026-07-24 15:26:44,119 INFO Query Execution Successful


True